# Geometry Simplification — Parameter Testing

Explores the three configurable steps in the `wkt.py` normalisation pipeline:

| Step | Parameter | Current value |
|------|-----------|---------------|
| `dump_wkt` round-trip | `dp` (decimal places) | 6 |
| `simplify()` | tolerance (degrees) | 0.000005 |
| `set_precision()` | grid size | 0.000001 |

---
**Grid analysis:** pivot tables showing area diff vs raw and vertex count across all
combinations. Cells are colour-coded: green = closer to raw / fewer vertices.

**Maps:** raw boundary always shown in dark blue. Toggle other layers on/off in the
top-right legend. Coloured outline = simplified boundary; filled area = symmetric
difference (where the boundary moved relative to raw).

In [6]:
import requests
import tempfile
import os

import pandas as pd
import geopandas as gpd
import shapely.wkt
from shapely import set_precision
from shapely.geometry import MultiPolygon
from shapely.geometry.polygon import orient
from shapely.validation import make_valid
import folium
from IPython.display import display, HTML

In [7]:
# Combos shown on the comparison maps.
# Each entry must have: label, dp, simplify (or None), precision_grid (or None), color.
PARAM_COMBOS = [
    {
        'label': 'Current (6dp, simplify=5e-6, grid=1e-6)',
        'dp': 6, 'simplify': 0.000005, 'precision_grid': 0.000001,
        'color': '#e74c3c',
    },
    {
        'label': 'No simplify (6dp, grid=1e-6)',
        'dp': 6, 'simplify': None, 'precision_grid': 0.000001,
        'color': '#2ecc71',
    },
    {
        'label': 'Light simplify (6dp, simplify=1e-6, grid=1e-6)',
        'dp': 6, 'simplify': 0.000001, 'precision_grid': 0.000001,
        'color': '#3498db',
    },
    {
        'label': 'No simplify, no grid snap (6dp)',
        'dp': 6, 'simplify': None, 'precision_grid': None,
        'color': '#1abc9c',
    },
    {
        'label': 'No simplify (7dp, grid=1e-7)',
        'dp': 7, 'simplify': None, 'precision_grid': 0.0000001,
        'color': '#9b59b6',
    },
]

In [8]:
def dump_wkt(geometry, precision=6, dimensions=2):
    wkt = shapely.wkt.dumps(geometry, rounding_precision=precision, output_dimension=dimensions)
    return wkt.replace(', ', ',')


def make_multipolygon(geometry):
    if geometry.geom_type in ['Point', 'Line', 'LineString', 'MultiLineString']:
        return None
    if geometry.geom_type == 'MultiPolygon':
        return geometry
    if geometry.geom_type == 'Polygon':
        return MultiPolygon([geometry])
    if geometry.geom_type == 'GeometryCollection':
        polygons = []
        for geom in geometry.geoms:
            if geom.geom_type == 'Polygon':
                polygons.append(geom)
            elif geom.geom_type == 'MultiPolygon':
                polygons.extend(geom.geoms)
            elif geom.geom_type == 'GeometryCollection':
                inner = make_multipolygon(geom)
                if inner:
                    polygons.extend(inner.geoms)
        return MultiPolygon(polygons)
    raise ValueError(f'unexpected geometry type: {geometry.geom_type}')


def apply_pipeline(raw_geom, simplify_tolerance=0.000005, dp=6, precision_grid=0.000001):
    # Replicates wkt.py normalise_geometry; pass None to skip a step.
    geom = shapely.wkt.loads(dump_wkt(raw_geom, precision=dp))

    if simplify_tolerance is not None and simplify_tolerance > 0:
        simplified = geom.simplify(simplify_tolerance)
        if not geom.is_valid or simplified.is_valid:
            geom = simplified

    if precision_grid is not None:
        geom = set_precision(geom, precision_grid, mode='pointwise')

    if not geom.is_valid:
        geom = make_valid(geom)

    geom = make_multipolygon(geom)

    if geom and not geom.is_valid:
        geom = geom.buffer(0)

    if geom:
        geom = make_multipolygon(geom)

    if geom:
        geom = MultiPolygon([orient(g) for g in geom.geoms])

    return geom


def vertex_count(geom):
    if geom is None:
        return 0
    if geom.geom_type == 'MultiPolygon':
        return sum(len(list(p.exterior.coords)) for p in geom.geoms)
    return len(list(geom.exterior.coords))


def compute_stats(raw_geom, processed_geom, label):
    raw_v = vertex_count(raw_geom)
    proc_v = vertex_count(processed_geom)
    area_diff = (
        processed_geom.symmetric_difference(raw_geom).area * 111000 * 70000
        if processed_geom
        else float('inf')
    )
    raw_size = len(raw_geom.wkt.encode())
    proc_size = len(processed_geom.wkt.encode()) if processed_geom else 0
    return {
        'Parameters': label,
        'Vertices': proc_v,
        'Vertex reduction %': round((1 - proc_v / raw_v) * 100, 1) if raw_v else 0,
        'Area diff vs raw (m2)': round(area_diff, 1),
        'WKT size reduction %': round((1 - proc_size / raw_size) * 100, 1) if raw_size else 0,
    }


def grid_analysis(raw_geom):
    simplify_vals  = [None, 0.000001, 0.0000025, 0.000005, 0.00001]
    dp_vals        = [5, 6, 7, 8]
    precision_vals = [None, 0.0000001, 0.000001, 0.00001]

    SLABELS = {
        None:     'None',
        0.000001: '1e-6',
        0.0000025:'2.5e-6',
        0.000005: '5e-6 (current)',
        0.00001:  '1e-5',
    }
    PLABELS = {
        None:       'None (skip)',
        0.0000001:  '1e-7',
        0.000001:   '1e-6 (current)',
        0.00001:    '1e-5',
    }
    S_ORDER  = ['None', '1e-6', '2.5e-6', '5e-6 (current)', '1e-5']
    DP_ORDER = ['5dp', '6dp', '7dp', '8dp']
    PG_ORDER = ['None (skip)', '1e-7', '1e-6 (current)', '1e-5']

    rows = []
    for s in simplify_vals:
        for dp in dp_vals:
            for pg in precision_vals:
                result = apply_pipeline(raw_geom, simplify_tolerance=s, dp=dp, precision_grid=pg)
                stats = compute_stats(raw_geom, result, '')
                rows.append({
                    'simplify': SLABELS[s],
                    'dp': f'{dp}dp',
                    'precision_grid': PLABELS[pg],
                    'area_diff_m2': stats['Area diff vs raw (m2)'],
                    'vertices': stats['Vertices'],
                })
    df = pd.DataFrame(rows)

    raw_v = vertex_count(raw_geom)
    display(HTML(f'<b>Raw vertices: {raw_v}</b>'))

    display(HTML('<h4>Area diff vs raw (m2) &mdash; simplify &times; dp &mdash; precision_grid = 1e-6</h4>'))
    pv1 = (
        df[df['precision_grid'] == '1e-6 (current)']
        .pivot(index='simplify', columns='dp', values='area_diff_m2')
        .reindex(S_ORDER)
        .reindex(columns=DP_ORDER)
    )
    display(pv1.style.format('{:.1f}').background_gradient(cmap='RdYlGn_r', axis=None))

    display(HTML('<h4>Vertex count &mdash; simplify &times; dp &mdash; precision_grid = 1e-6</h4>'))
    pv2 = (
        df[df['precision_grid'] == '1e-6 (current)']
        .pivot(index='simplify', columns='dp', values='vertices')
        .reindex(S_ORDER)
        .reindex(columns=DP_ORDER)
    )
    display(pv2.style.format('{:.0f}').background_gradient(cmap='RdYlGn_r', axis=None))

    display(HTML('<h4>Area diff vs raw (m2) &mdash; dp &times; precision_grid &mdash; simplify = None</h4>'))
    pv3 = (
        df[df['simplify'] == 'None']
        .pivot(index='dp', columns='precision_grid', values='area_diff_m2')
        .reindex(DP_ORDER)
        .reindex(columns=PG_ORDER)
    )
    display(pv3.style.format('{:.1f}').background_gradient(cmap='RdYlGn_r', axis=None))

    display(HTML('<h4>Vertex count &mdash; dp &times; precision_grid &mdash; simplify = None</h4>'))
    pv4 = (
        df[df['simplify'] == 'None']
        .pivot(index='dp', columns='precision_grid', values='vertices')
        .reindex(DP_ORDER)
        .reindex(columns=PG_ORDER)
    )
    display(pv4.style.format('{:.0f}').background_gradient(cmap='RdYlGn_r', axis=None))


def make_comparison_map(raw_geom, zoom=14):
    # One toggleable FeatureGroup per PARAM_COMBOS entry.
    # Raw boundary (dark blue) is always visible; all others start hidden.
    centroid = raw_geom.centroid
    m = folium.Map(
        location=[centroid.y, centroid.x],
        zoom_start=zoom,
        tiles='CartoDB positron',
    )

    fg_raw = folium.FeatureGroup(name='Raw (source)', show=True)
    folium.GeoJson(
        raw_geom.__geo_interface__,
        style_function=lambda x: {'color': '#1a237e', 'weight': 4, 'fillOpacity': 0.05},
        tooltip='Raw source geometry',
    ).add_to(fg_raw)
    fg_raw.add_to(m)

    for p in PARAM_COMBOS:
        result = apply_pipeline(
            raw_geom,
            simplify_tolerance=p['simplify'],
            dp=p['dp'],
            precision_grid=p['precision_grid'],
        )
        stats = compute_stats(raw_geom, result, p['label'])
        n_verts = stats['Vertices']
        a_diff = stats['Area diff vs raw (m2)']
        lbl = p['label']
        layer_name = f'{lbl}  ({n_verts} vertices, {a_diff} m2 diff)'
        fg = folium.FeatureGroup(name=layer_name, show=False)

        folium.GeoJson(
            result.__geo_interface__,
            style_function=lambda x, c=p['color']: {'color': c, 'weight': 2, 'fillOpacity': 0.0},
            tooltip=layer_name,
        ).add_to(fg)

        diff = result.symmetric_difference(raw_geom)
        if not diff.is_empty:
            folium.GeoJson(
                diff.__geo_interface__,
                style_function=lambda x, c=p['color']: {
                    'color': c, 'weight': 1, 'fillColor': c, 'fillOpacity': 0.45
                },
                tooltip=f'Changed area: {a_diff} m2',
            ).add_to(fg)

        fg.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    return m

In [9]:
url = 'https://open.barnet.gov.uk/download/20yo8/c6n/conservation_area.gpkg'
headers = {'User-Agent': 'Mozilla/5.0'}

with tempfile.NamedTemporaryFile(suffix='.gpkg', delete=False) as f:
    f.write(requests.get(url, headers=headers).content)
    tmp_path = f.name

barnet_data = gpd.read_file(tmp_path)
os.unlink(tmp_path)

totteridge_raw = barnet_data.loc[barnet_data['name'] == 'Totteridge'].geometry.iloc[0]
monken_raw     = barnet_data.loc[barnet_data['name'] == 'Monken Hadley'].geometry.iloc[0]
wood_raw       = barnet_data.loc[barnet_data['name'] == 'Wood Street'].geometry.iloc[0]

print(f'Loaded {len(barnet_data)} Barnet conservation areas')

Loaded 16 Barnet conservation areas


## Grid analysis

Each geometry gets four pivot tables:

1. **Area diff** (m2 vs raw) — simplify × dp — `precision_grid` held at current 1e-6
2. **Vertex count** — simplify × dp — same hold
3. **Area diff** — dp × `precision_grid` — `simplify` held at None (no simplification)
4. **Vertex count** — dp × `precision_grid` — same hold

Tables 3 and 4 address whether raising dp is useful if `set_precision` then snaps
coordinates back down (e.g. using 7dp but a 1e-6 grid throws away the 7th decimal place).

Green = lower (better for accuracy / smaller file). Red = higher.

### Totteridge — 2971 raw vertices

In [10]:
grid_analysis(totteridge_raw)

dp,5dp,6dp,7dp,8dp
simplify,,,,
None,2204.0,224.4,233.1,224.9
1e-6,2202.8,290.4,317.8,318.2
2.5e-6,2201.4,510.1,580.7,580.6
5e-6 (current),2323.1,1010.3,1074.1,1075.5
1e-5,2851.2,2142.7,2207.2,2205.4


dp,5dp,6dp,7dp,8dp
simplify,,,,
None,2629,2971,2971,2971
1e-6,1798,1023,944,940
2.5e-6,1608,686,668,670
5e-6 (current),1028,520,514,515
1e-5,444,381,378,377


precision_grid,None (skip),1e-7,1e-6 (current),1e-5
dp,,,,
5dp,2204.0,2204.0,2204.0,2210.3
6dp,224.4,224.4,224.4,2217.6
7dp,21.2,21.2,233.1,2209.6
8dp,2.2,21.8,224.9,2208.8


precision_grid,None (skip),1e-7,1e-6 (current),1e-5
dp,,,,
5dp,2629,2629,2629,2620
6dp,2971,2971,2971,2610
7dp,2971,2971,2971,2621
8dp,2971,2971,2971,2619


### Monken Hadley — 2192 raw vertices

In [11]:
grid_analysis(monken_raw)

dp,5dp,6dp,7dp,8dp
simplify,,,,
None,2703.0,282.4,281.9,283.8
1e-6,2703.1,376.9,398.4,401.1
2.5e-6,2763.1,716.1,761.5,761.5
5e-6 (current),2914.1,1390.6,1376.9,1384.5
1e-5,3514.5,2977.9,3036.1,3100.9


dp,5dp,6dp,7dp,8dp
simplify,,,,
None,2112,2192,2192,2192
1e-6,1597,1031,974,973
2.5e-6,1328,646,633,639
5e-6 (current),818,468,476,477
1e-5,390,331,331,327


precision_grid,None (skip),1e-7,1e-6 (current),1e-5
dp,,,,
5dp,2703.0,2703.0,2703.0,2704.3
6dp,282.4,282.4,282.4,2789.5
7dp,26.9,26.9,281.9,2721.2
8dp,2.7,26.7,283.8,2703.5


precision_grid,None (skip),1e-7,1e-6 (current),1e-5
dp,,,,
5dp,2112,2112,2112,2111
6dp,2192,2192,2192,2112
7dp,2192,2192,2192,2112
8dp,2192,2192,2192,2111


### Wood Street — 1196 raw vertices

In [12]:
grid_analysis(wood_raw)

dp,5dp,6dp,7dp,8dp
simplify,,,,
None,895.0,89.2,90.1,89.2
1e-6,893.5,118.8,130.3,129.7
2.5e-6,900.6,217.9,218.9,217.9
5e-6 (current),903.6,365.7,381.5,416.9
1e-5,1124.8,775.2,758.5,761.4


dp,5dp,6dp,7dp,8dp
simplify,,,,
None,1196,1196,1196,1196
1e-6,809,484,462,469
2.5e-6,701,325,325,327
5e-6 (current),460,271,266,261
1e-5,219,204,206,206


precision_grid,None (skip),1e-7,1e-6 (current),1e-5
dp,,,,
5dp,895.0,895.0,895.0,895.0
6dp,89.2,89.2,89.2,929.2
7dp,8.6,8.6,90.1,894.1
8dp,0.9,8.5,89.2,893.2


precision_grid,None (skip),1e-7,1e-6 (current),1e-5
dp,,,,
5dp,1196,1196,1196,1196
6dp,1196,1196,1196,1196
7dp,1196,1196,1196,1196
8dp,1196,1196,1196,1196


## Map comparison

Selected combinations from `PARAM_COMBOS` — edit that cell to change what appears here.
Toggle layers on/off in the legend; the filled colour shows where each setting moves the
boundary away from the raw source.

### Totteridge

In [13]:
make_comparison_map(totteridge_raw)

### Monken Hadley

In [14]:
make_comparison_map(monken_raw)

### Wood Street

In [15]:
make_comparison_map(wood_raw)